# Kim Flora · AI Voice Studio 0đ
Chạy ô 1, rồi ô 2. Dùng bảng điều khiển bên dưới; không dán lời vào code. Colab miễn phí có giới hạn. Mẫu giọng chỉ nằm trong phiên Colab.


In [ ]:
# @title 1. Cài công cụ trên Colab
%pip install -q vieneu ipywidgets soundfile


In [ ]:
# @title 2. Mở bảng điều khiển tạo giọng
import ipywidgets as w
from IPython.display import display, Audio, clear_output
from google.colab import files, output
from pathlib import Path
import subprocess, re, numpy as np, soundfile as sf, tempfile
from vieneu import Vieneu
output.enable_custom_widget_manager()
root=Path(tempfile.mkdtemp(prefix="kimflora_"))
state={"tts":None,"sample":None,"srt":None,"out":None}
def btn(label,style=""):
    return w.Button(description=label,button_style=style,layout=w.Layout(width="240px"))
sample=w.FileUpload(accept=".wav,.mp3,.m4a,.ogg,.flac",multiple=False,description="Chọn giọng")
srt=w.FileUpload(accept=".srt",multiple=False,description="Chọn SRT")
ref=w.Textarea(placeholder="Gõ đúng lời trong mẫu giọng nếu dùng chế độ chuẩn",layout=w.Layout(width="95%",height="85px"))
mode=w.Dropdown(options=[("Turbo v3 (không cần lời mẫu)","v3turbo"),("Chuẩn (cần lời mẫu)","standard")],description="Mô hình")
kind=w.ToggleButtons(options=[("File SRT","srt"),("Văn bản","text")],value="srt")
text=w.Textarea(placeholder="Nhập lời đọc tiếng Việt",layout=w.Layout(width="95%",height="120px"))
consent=w.Checkbox(description="Tôi có quyền sử dụng giọng mẫu",indent=False)
duration=w.FloatText(value=0,description="Dài clip (s)",layout=w.Layout(width="280px"))
make=btn("3. Tạo giọng AI","success")
download=btn("4. Tải WAV","info")
download.disabled=True
log=w.Output()
info=w.HTML("Chưa chọn mẫu giọng hoặc SRT.")
def status():
    info.value="Mẫu giọng: "+("đã chọn" if state["sample"] else "chưa chọn")+" · SRT: "+("đã chọn" if state["srt"] else "chưa chọn")
def runff(args):
    subprocess.run(["ffmpeg","-nostdin","-loglevel","error","-y",*map(str,args)],check=True)
def upload_one(widget):
    items=widget.value
    if len(items)!=1: raise ValueError("Chọn đúng một tệp.")
    item=next(iter(items.values())) if isinstance(items,dict) else items[0]
    return bytes(item["content"])
def load_sample(_):
    with log:
        try:
            raw=root/"sample.input"; raw.write_bytes(upload_one(sample))
            p=root/"reference.wav"
            runff(["-i",raw,"-ac",1,"-ar",24000,p])
            if sf.info(p).duration<2: raise ValueError("Mẫu quá ngắn.")
            state["sample"]=p; status()
            print("Mẫu giọng đã nhận. Nên dùng đoạn sạch 3–8 giây.")
            display(Audio(filename=str(p)))
        except Exception as e: print("Lỗi:",e)
def load_srt(_):
    with log:
        try:
            state["srt"]=upload_one(srt).decode("utf-8-sig")
            status(); print("Đã nhận SRT.")
        except Exception as e: print("Lỗi:",e)
def sec(t):
    h,m,s=t.replace(",",".").split(":")
    return int(h)*3600+int(m)*60+float(s)
def parse_srt(raw):
    rows=[]
    for block in re.split(r"\n\s*\n",raw.replace("\r","").strip()):
        lines=block.splitlines()
        for i,line in enumerate(lines):
            m=re.match(r"\s*(\d+:\d{2}:\d{2}[,.]\d{3})\s*-->\s*(\d+:\d{2}:\d{2}[,.]\d{3})",line)
            if m:
                a,b=map(sec,m.groups())
                t=re.sub(r"<[^>]+>","", " ".join(lines[i+1:])).strip()
                if t and b>a: rows.append((a,b,t))
                break
    if not rows: raise ValueError("Không tìm thấy phụ đề SRT hợp lệ.")
    return sorted(rows)
def synth(t,p):
    model=state["tts"]
    args={"text":t,"ref_audio":str(state["sample"])}
    if mode.value=="standard": args["ref_text"]=ref.value.strip()
    model.save(model.infer(**args),str(p))
def tempo_chain(r):
    parts=[]
    while r>2: parts.append(2);r/=2
    while r<0.5: parts.append(0.5);r/=0.5
    parts.append(r)
    return ",".join("atempo="+str(x) for x in parts)
def generate(_):
    with log:
        clear_output(wait=True)
        make.disabled=True; download.disabled=True; state["out"]=None
        try:
            if not consent.value or not state["sample"]: raise ValueError("Chọn mẫu giọng và xác nhận quyền sử dụng.")
            if mode.value=="standard" and not ref.value.strip(): raise ValueError("Nhập lời mẫu chính xác hoặc chọn Turbo v3.")
            rows=parse_srt(state["srt"] or "") if kind.value=="srt" else None
            if rows is None and not text.value.strip(): raise ValueError("Nhập lời đọc.")
            if rows and any(rows[i][1]>rows[i+1][0]+.001 for i in range(len(rows)-1)): raise ValueError("SRT có đoạn chồng thời gian. Sửa mốc trước khi tạo.")
            if state["tts"] is None or state.get("mode")!=mode.value:
                if state["tts"] is not None:
                    state["tts"].close(); state["tts"]=None
                print("Đang khởi tạo mô hình trên Colab…")
                state["tts"]=Vieneu(mode="v3turbo") if mode.value=="v3turbo" else Vieneu()
                state["mode"]=mode.value
            out=root/"kimflora-voice.wav"
            if not rows: synth(text.value.strip(),out)
            else:
                rate=24000
                end=max(rows[-1][1],duration.value)
                if end>7200: raise ValueError("Giới hạn mỗi lượt 2 giờ.")
                mix=np.zeros(int(np.ceil(end*rate)),dtype=np.float32)
                for i,(a,b,t) in enumerate(rows,1):
                    print(f"Đoạn {i}/{len(rows)} · {a:.2f}–{b:.2f}s",flush=True)
                    raw=root/"segment.wav"; fitted=root/"fitted.wav"
                    synth(t,raw)
                    n=sf.info(raw).duration
                    ratio=max(1,n/(b-a))
                    if ratio>1.8: print("  Cảnh báo: lời dài, đọc nhanh; nên rút gọn câu.")
                    runff(["-i",raw,"-af",tempo_chain(ratio),"-ac",1,"-ar",rate,fitted])
                    y,_=sf.read(fitted,dtype="float32")
                    lo=round(a*rate); hi=round(b*rate); length=min(len(y),hi-lo,len(mix)-lo)
                    if length>0:
                        if len(y)>length: print("  Cảnh báo: âm thanh vượt mốc; kiểm tra cuối câu.")
                        mix[lo:lo+length]=y[:length]
                sf.write(out,mix,rate,subtype="PCM_16")
            state["out"]=out;download.disabled=False
            print("Đã tạo WAV. Nghe thử trước khi tải.")
            display(Audio(filename=str(out)))
        except Exception as e: print("Lỗi:",type(e).__name__,str(e))
        finally: make.disabled=False
sample.observe(load_sample,names="value");srt.observe(load_srt,names="value");make.on_click(generate)
def save(_):
    if state["out"] and state["out"].exists(): files.download(str(state["out"]))
download.on_click(save)
display(w.VBox([w.HTML("<h2>KIM FLORA · AI VOICE STUDIO</h2><p>Chọn mẫu giọng, chọn SRT hoặc nhập văn bản, rồi tạo WAV. Không sửa mã Python.</p>"),w.HTML("<b>1. Tải mẫu giọng WAV / MP3 / M4A</b>"),sample,ref,mode,consent,kind,w.HTML("<b>2. Tải phụ đề SRT</b>"),srt,text,duration,info,make,download,log]))

